In [2]:
import os
import json
import time
import zipfile
import datetime
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
from utilis import Utility, Data, Visualization, Score

In [4]:
root_dir = os.getcwd()
print("Root directory is", root_dir)

Root directory is /Users/trungle/repo/Neurips2025_Weak_lensing


In [5]:
USE_PUBLIC_DATASET = True
DATA_DIR ='./dataset'  # This is only required when you set USE_PUBLIC_DATASET = True

In [6]:
# Initialize Data class object
data_obj = Data(data_dir=DATA_DIR, USE_PUBLIC_DATASET=USE_PUBLIC_DATASET)
data_obj.load_train_data()
# Load test data
data_obj.load_test_data()

Loading WIDE12H_bin2_2arcmin_kappa.npy: 100%|██████████| 11/11 [00:40<00:00,  3.65s/it]
Loading WIDE12H_bin2_2arcmin_kappa_noisy_test.npy: 100%|██████████| 40/40 [00:05<00:00,  7.74it/s]


In [6]:
Ncosmo = data_obj.Ncosmo
Nsys = data_obj.Nsys

print(f'There are {Ncosmo} cosmological models, each has {Nsys} realizations of nuisance parameters in the training data.')
print(f'Shape of the training data = {data_obj.kappa.shape}')
print(f'Shape of the mask = {data_obj.mask.shape}')
print(f'Shape of the training label = {data_obj.label.shape}')
print(f'Shape of the test data = {data_obj.kappa_test.shape}')

There are 101 cosmological models, each has 256 realizations of nuisance parameters in the training data.
Shape of the training data = (101, 256, 1424, 176)
Shape of the mask = (1424, 176)
Shape of the training label = (101, 256, 5)
Shape of the test data = (4000, 1424, 176)


In [7]:
# # Directory to save chunks
# !mkdir -p ./dataset/chunk_kappa
# save_dir = './dataset/chunk_kappa'
# os.makedirs(save_dir, exist_ok=True)

# # Number of chunks you want
# num_chunks = 10  # Change this as needed

# # Split kappa along the first axis (Ncosmo)
# chunks = np.array_split(data_obj.kappa, num_chunks, axis=0)

# for idx, chunk in enumerate(chunks):
#     chunk_path = os.path.join(save_dir, f'kappa_chunk_{idx}.npy')
#     np.save(chunk_path, chunk)
#     print(f'Saved {chunk_path}, shape: {chunk.shape}')

In [7]:
# Directory to save noisy chunks
!mkdir -p ./dataset/chunk_kappa_noise
save_dir = './dataset/chunk_kappa_noise'
os.makedirs(save_dir, exist_ok=True)

# Number of chunks
num_chunks = 10  # Adjust as needed

np.random.seed(113)  # For reproducibility

chunks = np.array_split(data_obj.kappa, num_chunks, axis=0)

for idx, chunk in enumerate(chunks):
    noisy_chunk = Utility.add_noise(
        data=chunk.astype(np.float64),
        mask=data_obj.mask,
        ng=data_obj.ng,
        pixel_size=data_obj.pixelsize_arcmin
    )
    chunk_path = os.path.join(save_dir, f'kappa_noisy_chunk_{idx}.npy')
    np.save(chunk_path, noisy_chunk)
    print(f'Saved {chunk_path}, shape: {noisy_chunk.shape}')

Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_0.npy, shape: (11, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_1.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_2.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_3.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_4.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_5.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_6.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_7.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_8.npy, shape: (10, 256, 1424, 176)
Saved ./dataset/chunk_kappa_noise/kappa_noisy_chunk_9.npy, shape: (10, 256, 1424, 176)


In [9]:
import numpy as np
import os

# Đường dẫn tới thư mục chứa các chunk noisy và file label gốc
chunk_dir = './dataset/chunk_kappa_noise'
label_path = './dataset/label.npy'

# Lấy danh sách các file chunk noisy, đảm bảo đúng thứ tự
chunk_files = sorted([f for f in os.listdir(chunk_dir) if f.startswith('kappa_noisy_chunk_') and f.endswith('.npy')])

# Nạp toàn bộ label
labels = np.load(label_path)

start = 0
for chunk_file in chunk_files:
    chunk_path = os.path.join(chunk_dir, chunk_file)
    noisy_chunk = np.load(chunk_path)
    num_samples = noisy_chunk.shape[0]
    labels_chunk = labels[start:start+num_samples]
    
    # Lưu file nhãn tương ứng
    label_chunk_name = chunk_file.replace('kappa_noisy_chunk_', 'label_chunk_')
    label_chunk_path = os.path.join(chunk_dir, label_chunk_name)
    np.save(label_chunk_path, labels_chunk)
    
    print(f"Saved {label_chunk_path} with shape {labels_chunk.shape}")
    start += num_samples

Saved ./dataset/chunk_kappa_noise/label_chunk_0.npy with shape (11, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_1.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_2.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_3.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_4.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_5.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_6.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_7.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_8.npy with shape (10, 256, 5)
Saved ./dataset/chunk_kappa_noise/label_chunk_9.npy with shape (10, 256, 5)
